In [7]:
#Imports
from google.colab import drive
drive.mount('/content/drive')
import os
import os.path as osp
import json
import pandas as pd
import numpy as np


Mounted at /content/drive


In [8]:
dataset_path = "/content/drive/MyDrive/pads-parkisons-dataset-folder"
os.listdir(dataset_path)

['pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip']

In [9]:


zip_path = "/content/drive/MyDrive/pads-parkisons-dataset-folder/pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip"
extract_path = "/content/pads_dataset"

import zipfile
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

os.listdir(extract_path)

['pads-parkinsons-disease-smartwatch-dataset-1.0.0']

In [10]:
base_path = f"{extract_path}/pads-parkinsons-disease-smartwatch-dataset-1.0.0"
os.listdir(base_path)

['preprocessed',
 'SHA256SUMS.txt',
 'scripts',
 'movement',
 'patients',
 'LICENSE.txt',
 'questionnaire']

In [11]:
# file paths
base_path = f"{extract_path}/pads-parkinsons-disease-smartwatch-dataset-1.0.0"
timeseries_dir = f"{base_path}/movement/timeseries"
observation_dir = f"{base_path}/movement"
patients_dir = f"{base_path}/patients"

In [12]:
# load the timeseries file
def load_timeseries(filepath):
    columns = ["Time", "Accel_X", "Accel_Y", "Accel_Z", "Gyro_X", "Gyro_Y", "Gyro_Z"]
    df = pd.read_csv(filepath, header=None, names=None)
    return df

In [13]:
filepath = f"{timeseries_dir}/001_Relaxed_LeftWrist.txt"
df = load_timeseries(filepath)
print(df.head(5))
print(df.shape)

          0         1         2             3         4         5         6
0  0.000000 -0.003958  0.002360  1.746120e-03 -0.015492  0.005616  0.001034
1  0.009903 -0.004005  0.002259  8.355394e-04 -0.012276  0.005605 -0.003218
2  0.019901  0.000833  0.002225  9.437218e-04 -0.009051  0.002432 -0.002137
3  0.029907 -0.000191  0.004109  4.783000e-07 -0.008911  0.002154 -0.000931
4  0.039984  0.001769  0.004064 -1.902400e-03 -0.007830  0.000034  0.000145
(2048, 7)


In [14]:

def get_patient_label(patient_id,patients_dir):
  individual_file= f"patient_{patient_id:03d}.json" # the padding ensures we can enter an int without thinking about the zeros
  file_path=osp.join(patients_dir,individual_file)

  # reading the file contents
  with open(file_path) as file:
    contents = json.load(file)
    #  get the condition for each patient
    condition= contents.get("condition")

    return condition


In [17]:
label= get_patient_label(1,patients_dir)
print(label)

Healthy


In [15]:
def build_manifest(patients_dir, observation_dir, n_patients=469):
    rows = []
    for pat_num in range(1, n_patients + 1):
        patient_id = f"{pat_num:03d}"
        label = get_patient_label(pat_num, patients_dir)

        obs_path = osp.join(observation_dir, f"observation_{patient_id}.json")
        with open(obs_path) as f:
            obs_contents = json.load(f)

        for session in obs_contents["session"]:
            activity_name = session.get("record_name")
            for record in session['records']:
                left_right_wrist = record.get('device_location' )
                filename = record.get('file_name')
                rows.append({
                    "patient_id": patient_id,
                    "label": label,
                    "task": activity_name,
                    "wrist": left_right_wrist,
                    "filepath": filename
                })

    return pd.DataFrame(rows)


full_path = osp.join(base_path, "movement", row["filepath"])
df = load_timeseries(full_path)

In [18]:
manifest = build_manifest(patients_dir, observation_dir, n_patients=469)
print(manifest.to_string(index=False))
print(manifest.shape)          # expected (469 * 11 * 2, 5) = (10318, 5)
print(manifest.label.value_counts())   # expected roughly: 291 PD-ish, 99 DD-ish, 79 HC-ish subjects worth of rows
print(manifest.isnull().sum()) # any column with nulls = a real problem to chase down
manifest.to_csv("/content/drive/MyDrive/pads-parkisons-dataset-folder/manifest.csv", index=False)



patient_id                    label        task      wrist                                  filepath
       001                  Healthy     Relaxed  LeftWrist      timeseries/001_Relaxed_LeftWrist.txt
       001                  Healthy     Relaxed RightWrist     timeseries/001_Relaxed_RightWrist.txt
       001                  Healthy RelaxedTask  LeftWrist  timeseries/001_RelaxedTask_LeftWrist.txt
       001                  Healthy RelaxedTask RightWrist timeseries/001_RelaxedTask_RightWrist.txt
       001                  Healthy StretchHold  LeftWrist  timeseries/001_StretchHold_LeftWrist.txt
       001                  Healthy StretchHold RightWrist timeseries/001_StretchHold_RightWrist.txt
       001                  Healthy    LiftHold  LeftWrist     timeseries/001_LiftHold_LeftWrist.txt
       001                  Healthy    LiftHold RightWrist    timeseries/001_LiftHold_RightWrist.txt
       001                  Healthy  HoldWeight  LeftWrist   timeseries/001_HoldWeight_Left